In [ ]:
import pandas as pd  # imports the pandas library, used for data analysis and handling CSV/Excel files


In [ ]:
df = pd.read_csv("C:/Users/jovit/Desktop/Netflix_Project/netflix_titles.csv")
# loads the Netflix dataset CSV into a pandas DataFrame called df
# DataFrame = table-like structure (rows & columns) for easy analysis

In [3]:
df.head()        # shows first 5 rows
df.info()        # shows column names and data types
df.shape         # shows (rows, columns)


<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


(8807, 12)

In [ ]:
print(df.columns.tolist())
# prints all column names from the DataFrame as a Python list
# useful to quickly see the structure of the dataset (headers)

['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']


In [ ]:
table_name = "netflix"  

# Handle reserved keywords (avoid SQL errors)
reserved = {"cast": "cast_name", "type": "show_type"}  
cols = [reserved.get(c, c) for c in df.columns]   # replace reserved names if found

# Generate CREATE TABLE script automatically
sql = f"CREATE TABLE {table_name} (\n"
sql += ",\n".join([f'"{c}" TEXT' for c in cols])  # define each column as TEXT
sql += "\n);"

print(sql)   # prints the CREATE TABLE statement for PostgreSQL


CREATE TABLE netflix (
"show_id" TEXT,
"show_type" TEXT,
"title" TEXT,
"director" TEXT,
"cast_name" TEXT,
"country" TEXT,
"date_added" TEXT,
"release_year" TEXT,
"rating" TEXT,
"duration" TEXT,
"listed_in" TEXT,
"description" TEXT
);


In [ ]:
pip install psycopg2-binary
# installs the psycopg2-binary package, a PostgreSQL adapter for Python
# lets Python connect to PostgreSQL databases and run SQL queries

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.8/2.8 MB 8.5 MB/s eta 0:00:01
   ---------------------- ----------------- 1.6/2.8 MB 5.0 MB/s eta 0:00:01
   --------------------------------- ------ 2.4/2.8 MB 4.5 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 4.4 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Users\jovit\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [ ]:
import psycopg2
# imports the psycopg2 library, which lets Python connect to PostgreSQL databases
# you can then use psycopg2.connect() to open a connection and run SQL queries

In [ ]:
conn = psycopg2.connect(
    dbname="netflix_db",   # your database name
    user="postgres",       # your username
    password="your_password",  # your password
    host="localhost",      # server address
    port="5432"            # default PostgreSQL port
)
cur = conn.cursor()        # creates a cursor to execute SQL commands

In [ ]:
# Create table in PostgreSQL
cur.execute(sql)   # runs the CREATE TABLE statement generated earlier
conn.commit()      # saves (commits) the change to the database

DuplicateTable: relation "netflix" already exists


In [ ]:
conn.rollback()
# undoes the current transaction if something went wrong
# useful when a SQL command fails and leaves the session in an aborted state

In [ ]:
# 🔹 Step 4: Import CSV into PostgreSQL
with open("C:/Users/jovit/Desktop/Netflix_Project/netflix_titles.csv", "r", encoding="utf-8") as f:
    # opens the CSV file in read mode with UTF-8 encoding
    cur.copy_expert(
        f"COPY {table_name} FROM STDIN WITH CSV HEADER DELIMITER ','", f
    )
    # executes PostgreSQL COPY command to bulk load data into the table
    # CSV HEADER → skips the first row (column names)
    # DELIMITER ',' → specifies comma-separated values

conn.commit()  
# commits the transaction so the imported data is permanently saved

In [ ]:
# 🔹 Step 5: Verify data
cur.execute(f"SELECT COUNT(*) FROM {table_name};")  
# runs a SQL query to count all rows in the table

print("Row count:", cur.fetchone()[0])  
# fetches the count result and prints it

cur.execute(f"SELECT * FROM {table_name} LIMIT 5;")  
# retrieves the first 5 rows from the table for sampling

print("Sample rows:", cur.fetchall())  
# prints the sample rows as a list of tuples

cur.close()  
conn.close()  
# closes the cursor and connection to free resources

print("✅ Table created and data imported successfully!")  
# confirmation message for successful import

Row count: 8807
Sample rows: [('s1', 'Movie', 'Dick Johnson Is Dead', 'Kirsten Johnson', None, 'United States', 'September 25, 2021', '2020', 'PG-13', '90 min', 'Documentaries', 'As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.'), ('s2', 'TV Show', 'Blood & Water', None, 'Ama Qamata, Khosi Ngema, Gail Mabalane, Thabang Molaba, Dillon Windvogel, Natasha Thahane, Arno Greeff, Xolile Tshabalala, Getmore Sithole, Cindy Mahlangu, Ryle De Morny, Greteli Fincham, Sello Maake Ka-Ncube, Odwa Gwanya, Mekaila Mathys, Sandi Schultz, Duane Williams, Shamilla Miller, Patrick Mofokeng', 'South Africa', 'September 24, 2021', '2021', 'TV-MA', '2 Seasons', 'International TV Shows, TV Dramas, TV Mysteries', 'After crossing paths at a party, a Cape Town teen sets out to prove whether a private-school swimming star is her sister who was abducted at birth.'), ('s3', 'TV Show', 'Ganglands', 'Julien Leclerc

In [ ]:
movies = ["Inception", "The Irishman", "Extraction"]  
# creates a Python list (array) with three movie titles

print(movies[0])   
# prints the first element of the list → "Inception"


Inception


In [ ]:
class Node:
    def __init__(self, data):
        self.data = data        # stores the movie title
        self.next = None        # pointer to the next node

class LinkedList:
    def __init__(self):
        self.head = None        # start of the linked list

    def append(self, data):
        new_node = Node(data)   # create a new node
        if not self.head:       # if list is empty, set head
            self.head = new_node
            return
        last = self.head
        while last.next:        # traverse until the last node
            last = last.next
        last.next = new_node    # link new node at the end

    def display(self):
        current = self.head
        while current:          # traverse from head to end
            print(current.data) # print each movie title
            current = current.next

# Example usage
ll = LinkedList()
ll.append("Inception")
ll.append("The Irishman")
ll.append("Extraction")
ll.display()   # prints all movies in sequence

Inception
The Irishman
Extraction


In [ ]:
import psycopg2

# Connect to PostgreSQL database
conn = psycopg2.connect(
    dbname="netflix_db",
    user="postgres",
    password="jovita",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

# Run SQL query: select PG-13 Drama movies
cur.execute("""
    SELECT title
    FROM netflix
    WHERE show_type = 'Movie'
      AND listed_in LIKE '%Drama%'
      AND rating = 'PG-13';
""")

titles = [row[0] for row in cur.fetchall()]  
# fetch all matching rows, store only the title column in a list

conn.close()  
# close database connection

# Store results in linked list
ll = LinkedList()
for t in titles:
    ll.append(t)   # append each movie title as a node

ll.display()  
# print all movie titles sequentially from the linked list

The Starling
Birth of the Dragon
Worth
Agatha Christie's Crooked House
An Unfinished Life
Clear and Present Danger
Dear John
Freedom Writers
Marshall
Poseidon
Same Kind of Different as Me
Like Crazy
Quartet
Catch Me If You Can
Five Feet Apart
My Girl
Seabiscuit
Space Cowboys
The Book of Henry
The Twilight Saga: Breaking Dawn: Part 1
The Twilight Saga: Breaking Dawn: Part 2
The Twilight Saga: Eclipse
The Twilight Saga: New Moon
Twilight
Home Again
Midnight Sun
Hampstead
Life as We Know It
Memoirs of a Geisha
Mother's Day
Ophelia
Seven Pounds
The Best of Enemies
The Queen
What Dreams May Come
Fatherhood
Lowriders
2 Hearts
I Am Sam
Million Dollar Baby
The Meddler
The Mirror Has Two Faces
Collateral Beauty
Never Back Down
State of Play
The Lovely Bones
The Zookeeper's Wife
3 Idiots
At Eternity's Gate
Mandela: Long Walk to Freedom
Philomena
Get on Up
Bombay Rose
Safe Haven
Moxie
Black or White
Dances with Wolves
Operation Finale
Fatima
The Dig
The Next Three Days
A Monster Calls
Gimme Shelt